* 기본 설정

In [1]:
import os
import pathlib

here = pathlib.Path.cwd()
ROOT = here.parents[2] if here.name == "day04" else here
os.chdir(ROOT)
SANDBOX = ROOT / "sandbox" / "w3" / "day04"

print("프로젝트 루트  :", ROOT)

프로젝트 루트  : /Users/pyoyoung-gyu/Desktop/Personal Project/한화아카데미/AI 서비스 백엔드 프로그래밍 실무/hanwha-agent/agent_practice


In [2]:
import anthropic
import sys

print("anthropic 버전 : ", anthropic.__version__)
print("지금의 파이썬 커널 버전 ", sys.executable)

anthropic 버전 :  1.6.0
지금의 파이썬 커널 버전  /Users/pyoyoung-gyu/Desktop/Personal Project/한화아카데미/AI 서비스 백엔드 프로그래밍 실무/hanwha-agent/agent_practice/.venv/bin/python


In [3]:
import sys

if str(ROOT / 'backend') not in sys.path:
    sys.path.insert(0, str(ROOT / 'backend'))

In [4]:
from app.core.config import get_settings, mask

settings = get_settings()
print('모드        :', settings.app_mode)
print('모델        :', settings.llm_model)

if settings.anthropic_api_key is None:
    print('API 키      : (없음) — .env 의 ANTHROPIC_API_KEY 가 비어 있습니다')
else:
    print('API 키      :', mask(settings.anthropic_api_key.get_secret_value()))


모드        : mock
모델        : claude-haiku-4-5
API 키      : sk-ant-a...(108자)


In [5]:
import inspect

from anthropic.resources.messages import Messages

# Message.creat : API 한테 보내줄 메시지 작성하는 기능
params = inspect.signature(Messages.create).parameters
# vlftn vkfkalxj whghl
required = [name for name, p in params.items() if p.default is inspect.Parameter.empty and name != "self"]
print(required)

QUESTION = "제주도 출장 숙박비 한도가 얼마인가요?"

try:
    Messages.create(
        None, # client가 와야하는 자리
        model="clauded-haiku-4-5",
        # max_tokens 입력 안하면 TypeError
        messages = [
            {"role" : "User", "content" : QUESTION}
        ],
    )
except TypeError as exc:
    print(type(exc).__name__)
    print(exc)

['max_tokens', 'messages', 'model']
TypeError
Missing required arguments; Expected either ('max_tokens', 'messages' and 'model') or ('max_tokens', 'messages', 'model' and 'stream') arguments to be given


* 금액 계산기

In [6]:
PRICING = {'input': 1.0, 'cache_write': 1.25, 'cache_read': 0.1, 'output': 5.0}
USD_KRW = 1400.0

def cost_krw(input_tok: int, output_tok: int) -> float:
    usd = (input_tok * PRICING['input'] + output_tok * PRICING['output']) / 1000000
    return round(usd * USD_KRW, 1)

total_input = 69
output_tok = 146

(WED_INPUT, WED_OUTPUT) = (800, 400)
wed = cost_krw(WED_INPUT, WED_OUTPUT) # 가짜 금액
today = cost_krw(total_input, output_tok)
print(f'어제 어림한 값 : {wed } 원   (입력 {WED_INPUT} · 출력 {WED_OUTPUT} 토큰)')
print(f'오늘 실제 호출     : {today} 원   (입력 {total_input} · 출력 {output_tok} 토큰)')
print(f'차이               : {round(wed - today, 1)} 원')

어제 어림한 값 : 3.9 원   (입력 800 · 출력 400 토큰)
오늘 실제 호출     : 1.1 원   (입력 69 · 출력 146 토큰)
차이               : 2.8 원


* 포트 확인

In [8]:
# 파이썬은 한번 훑은 폴더의 파일 목록을 기억해둔다.
# 노트북 처럼 돌면서 파일을 새로 만드는 자리에서는, 방금 만든 파일을 못 볼 수 있다.
# 아래 invalidate_caches()로 그 기억을 비워서 다시 훑게 만들기.
import importlib
importlib.invalidate_caches()


from app.integrations.ports import LLMPort, LLMResult

result = LLMResult(text="1박 70,000원 이내", model="claude-haiku-4-5")
print(result)
print("기본값 확인")
print("input_token: ", result.input_tok)
print("extras: ", result.extras)
print("cost_krw: ", result.cost_krw)
print("cost_krw: ", LLMPort.__name__)

LLMResult(text='1박 70,000원 이내', model='claude-haiku-4-5', input_tok=0, cache_tok=0, output_tok=0, cost_krw=0.0, latency_ms=0, extras={})
기본값 확인
input_token:  0
extras:  {}
cost_krw:  0.0
cost_krw:  LLMPort


In [9]:
import importlib
importlib.invalidate_caches() 

from app.integrations.llm_claude import USD_KRW, estimate_cost_krw

input_tok = 62
output_tok = 118

print("환율 : ", USD_KRW)
print("계산한 비용 : ", estimate_cost_krw(input_tok, output_tok), "원")

환율 :  1400.0
계산한 비용 :  0.9 원


In [10]:
import importlib
import inspect
from app.integrations import llm_claude

llm_claude = importlib.reload(llm_claude)

lines = inspect.getsource(llm_claude.ClaudeLLM.__init__).splitlines()

(shown, inside_doc) = ([], False)

for line in lines:
    if line.strip().startswith('"""'):
        inside_doc = not inside_doc
        continue
    if not inside_doc:
        shown.append(line)
        
print('── ClaudeLLM.__init__ 소스 (docstring 은 접었다) ──')
print('\n'.join(shown))
print('answer 있는가:', hasattr(llm_claude.ClaudeLLM, 'answer'))


── ClaudeLLM.__init__ 소스 (docstring 은 접었다) ──
    def __init__(self) -> None:
        try:
            from anthropic import Anthropic
        except ImportError as exc:
            raise ExternalServiceError(
                "anthropic 패키지가 설치 되어 있지 않습니다."
            ) from exc

        settings = get_settings()
        key = settings.anthropic_api_key
        if key is None:
            raise ExternalServiceError("ANTHROPIC_API_KEY 가 비어있습니다.")
        self._client = Anthropic(api_key=key.get_secret_value())
        self.__model = settings.llm_model
answer 있는가: True


In [11]:
import importlib
import inspect
from app.integrations import llm_claude, ports
llm_claude = importlib.reload(llm_claude)
sig_port = inspect.signature(ports.LLMPort.answer, eval_str=True)
sig_impl = inspect.signature(llm_claude.ClaudeLLM.answer, eval_str=True)
print('LLMPort.answer   :', sig_port)
print('ClaudeLLM.answer :', sig_impl)
print()
print('_load_prompt 있는가   :', hasattr(llm_claude, '_load_prompt'), '  ← 노트북 02 에서 만든다')
print('_context_block 있는가 :', hasattr(llm_claude, '_context_block'), '  ← 노트북 03 에서 만든다')

LLMPort.answer   : (self, *, question: str, contexts: list[dict], user: dict) -> app.integrations.ports.LLMResult
ClaudeLLM.answer : (self, *, question: str, contexts: list[dict], user: dict) -> app.integrations.ports.LLMResult

_load_prompt 있는가   : True   ← 노트북 02 에서 만든다
_context_block 있는가 : True   ← 노트북 03 에서 만든다


In [10]:
import importlib
importlib.invalidate_caches()

from app.core.config import get_settings
from app.core.exceptions import ModeNotAvailable
from app.integrations.factory import get_llm

print('is_live :', get_settings().is_live)
try:
    llm = get_llm()
    print('어댑터 :', llm.name)
except ModeNotAvailable as exc:
    print(f'{type(exc).__name__} ({exc.status_code})')
    print(exc.message)

is_live : False
ModeNotAvailable (409)
테스트용 mock 어댑터는 만들지 않았습니다..env의 APP_MODE를 live로 두고 터미넬에서 부르세요.


In [11]:
from pathlib import Path

P = Path("backend") / "app" / "agent" / "prompts" / "answer_system.md"
text = P.read_text(encoding="utf-8")

print("존재 여부 : ", P.exists())
print("첫 두 줄 : ", text.splitlines()[:2])
print("글자 수 : ", len(text))

존재 여부 :  True
첫 두 줄 :  ['당신은 사내 규정 질의응답 도우미 입니다.', '']
글자 수 :  1102


In [12]:
import importlib
import sys
from pathlib import Path

BACKEND = Path.cwd() / 'backend'
if str(BACKEND) not in sys.path:
    sys.path.insert(0, str(BACKEND))
    
import app.integrations.llm_claude as llm_claude
importlib.reload(llm_claude)

from app.integrations.llm_claude import PROMPTS, _load_prompt

print('PROMPTS 경로 :', PROMPTS)

good = _load_prompt('answer_system.md')
bad = _load_prompt('answer_system.md')

print('정상 이름 : 글자 수', len(good), '->', repr(good.splitlines()[0]))
print('오타 이름 : 글자 수', len(bad), '  ->', repr(bad))


PROMPTS 경로 : /Users/pyoyoung-gyu/Desktop/Personal Project/한화아카데미/AI 서비스 백엔드 프로그래밍 실무/hanwha-agent/agent_practice/backend/app/agent/prompts
정상 이름 : 글자 수 1102 -> '당신은 사내 규정 질의응답 도우미 입니다.'
오타 이름 : 글자 수 1102   -> '당신은 사내 규정 질의응답 도우미 입니다.\n\n## 역할\n\n한화시스템 임직원이 사내 규정에 관해 물으면, 함께 주어진 근거 문서만 읽고 답합니다.\n등록된 근거 문서는 다음 세 건입니다.\n\n| 문서 번호 | 제목 | 버전 | 소관 | 보안등급 |\n|---|---|---|---|---|\n| DOC-HR-014 | 국내출장 여비 규정 | v2.0 | 인사총무 | 일반 |\n| DOC-PU-007 | 구매·계약 규정 | v4.0 | 구매팀 | 대외비 |\n| DOC-SE-003 | 정보보안 지침 | v2.2 | 보안팀 | 대외비 |\n\n## 규칙\n\n1. 주어진 근거 문서 밖의 내용을 지어내지 않습니다. 모르면 모른다고 말합니다.\n2. 근거가 부족하면 부족하다고 먼저 밝힙니다. 다른 문서의 내용으로 유추해서 메우지 않습니다.\n3. 열람 권한이 없는 문서는 인용하지 않습니다. 보안등급은 일반·3급·대외비 세 단계입니다.\n4. 금액·기한·조건은 근거에 적힌 숫자를 그대로 옮깁니다. 계산이 필요하면 계산 과정을 보입니다.\n5. "아마", "일반적으로" 같은 표현으로 빈틈을 메우지 않습니다.\n\n## 출력 형식\n\n아래 세 필드를 가진 JSON 하나만 보냅니다.\n\n| 필드 | 형 | 뜻 |\n|---|---|---|\n| `answer` | 문자열 | 한국어 답변 본문. 결론을 먼저 씁니다 |\n| `sources` | 목록 | 인용한 근거. 항목마다 `doc_id` · `title` · `version` · `locator` |\

In [13]:
CONTEXTS = [
    {
        "doc_id": "DOC-HR-014",
        "title": "국내출장 여비 규정",
        "version": "v2.0",
        "locator": "제12조(숙박비) · p.6",
        "quote": "서울·광역시: 1박 70,000원 이내",
        "score": 0.91,
    },
    {
        "doc_id": "DOC-PU-007",
        "title": "구매·계약 규정",
        "version": "v4.0",
        "locator": "제7조 · p.3",
        "quote": "출장 중 물품 구매는 사전 품의를 원칙으로 한다.",
        "score": 0.62,
    },
    {
        "doc_id": "DOC-SE-003",
        "title": "정보보안 지침",
        "version": "v2.2",
        "locator": "제4조 · p.2",
        "quote": "대외비 문서는 열람 권한이 확인된 임직원에게만 제공한다.",
        "score": 0.48,
    },
]


In [14]:
def context_block(contexts: list[dict]) -> str:
    lines = []
    for i, c in enumerate(contexts, 1):
        lines.append(
            f"[근거 {i}] {c.get('title')} {c.get('version')} · {c.get('locator')} "
            f"(유사도 {c.get('score', 0):.2f})\n{c.get('quote') or c.get('text') or ''}"
        )
    return "\n\n".join(lines) if lines else "(근거 문서 없음)"


print(context_block(CONTEXTS))
print()
print("── 근거가 0건일 때 ──")
print(context_block([]))

[근거 1] 국내출장 여비 규정 v2.0 · 제12조(숙박비) · p.6 (유사도 0.91)
서울·광역시: 1박 70,000원 이내

[근거 2] 구매·계약 규정 v4.0 · 제7조 · p.3 (유사도 0.62)
출장 중 물품 구매는 사전 품의를 원칙으로 한다.

[근거 3] 정보보안 지침 v2.2 · 제4조 · p.2 (유사도 0.48)
대외비 문서는 열람 권한이 확인된 임직원에게만 제공한다.

── 근거가 0건일 때 ──
(근거 문서 없음)


In [15]:
import importlib
import app.integrations.llm_claude as llm_claude
importlib.reload(llm_claude)

from app.integrations.llm_claude import _extract_json
from app.schemas.chat import AnswerOut
from pydantic import ValidationError

# 모델이 json 형식의 응답을 ``` 백틱으로 감싸서 주는 경우 (예시)
RAW = (
    "```json\n"
    '{"answer":"1박 70,000원 이내",'
    '"sources":[{"doc_id":"DOC-HR-014","title":"국내출장 여비 규정",'
    '"version":"v2.0","locator":"제12조(숙박비) · p.6"}],'
    '"enough_evidence":true}\n'
    "```"
)

try:
    AnswerOut.model_validate_json(RAW)
except ValidationError as e:
    print('① 그대로 넣으면 :', e.errors()[0]['msg'])

# 모델이 답변한 json을 전처리 하고
data = _extract_json(RAW)
# 체크하면
a = AnswerOut.model_validate(data)

print(f'② 벗기고 검증   : answer={a.answer!r} · 근거 {len(a.sources)}건 · 충분함 {a.enough_evidence}')
print('③ JSON 이 아니면 :', _extract_json('죄송합니다. 답변을 만들지 못했습니다.'))


① 그대로 넣으면 : Invalid JSON: expected value at line 1 column 1
② 벗기고 검증   : answer='1박 70,000원 이내' · 근거 1건 · 충분함 True
③ JSON 이 아니면 : {}
